# YAPEAL data · first look and a case

Part 1: what's in the data. Part 2: the case I'd go for (wallet leakage, stakeholder YAPEAL).

Reads the raw files from `data/raw/`. Everything below is aggregated, but it comes from the NDA data. Repo stays private, outputs are stripped on commit (nbstripout).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# repo root, whether the notebook runs from notebooks/ or from the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "raw"

# fixed colour order, never cycled
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
INK2, GRID = "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "figure.dpi": 110,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False,
})
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)


def clean(ax, title, ylabel=""):
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.grid(axis="x", visible=False)


def label_end(ax, x, series, name, color):
    ax.plot(x, series, color=color, lw=2)
    ax.annotate(name, (x[-1], series.iloc[-1]), xytext=(5, 0), textcoords="offset points", va="center", color=INK2)

In [ ]:
def load_sow(name):
    df = pd.read_csv(f"{DATA}/{name}").rename(columns={"n_transasctions": "n_transactions"})
    df["date"] = pd.to_datetime(df["year_month"], format="%m-%Y")
    # categories come in two spellings, merge them
    df["category"] = df["category"].str.lower().replace({"restaurant": "restaurants", "lebensmittel": "groceries"})
    if "top_counterpart" in df:
        df["top_counterpart"] = df["top_counterpart"].replace(
            {"brezelkönig": "brezelkonig", "mcdonald's": "mcdonalds", "amzn": "amazon"})
    df = df[df["date"] >= "2021-01-01"]  # 2020-12 is one customer
    df["quarter"] = df["date"].dt.to_period("Q")
    return df


sow = load_sow("sow_category.csv")
cp = load_sow("sow_category_counterpart.csv")
cust = pd.read_csv(f"{DATA}/customer_data.csv")
labels = pd.read_csv(f"{DATA}/customer_data_labels.csv")
predict = pd.read_csv(f"{DATA}/customer_data_predict.csv")

CAT = [c for c in cust if c.startswith("cat_")]
CUR = [c for c in cust if c.startswith("cur_")]
CTY = [c for c in cust if c.startswith("country_")]

print("customers      ", cust.shape)
print("labels         ", labels.shape, "churn rate", round(labels["churned"].mean(), 3))
print("to predict     ", predict.shape)
print("sow category   ", sow.shape, sow["date"].min().date(), "→", sow["date"].max().date())
print("sow counterpart", cp.shape, cp["top_counterpart"].nunique(), "counterparts")

## 1. what we have

Three files, three grains, no join between them except the category name.

- `customer_data.csv` – one row per customer. totals, tenure, POS/e-com, spend per category / currency / country. **no time, no counterparts**
- `customer_data_labels.csv` / `_predict.csv` – churn label for 70 %, predict the rest. hand-in 14.10
- `sow_category.csv` – month × category
- `sow_category_counterpart.csv` – month × category × top counterpart (≥ 1000 tx, rest = `other`). **no customers**

So: anything about Coop / Revolut / SBB is a 36-point time series. Anything about segments or churn is customer level and only knows categories.

In [ ]:
monthly = sow.groupby("date").agg(active=("n_customers", "max"), spend=("total_amount", "sum"))
monthly["spend_per_active"] = monthly["spend"] / monthly["active"]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(monthly.index, monthly["active"], color=C[0], lw=2)
clean(axes[0], "monthly active customers (max over categories)", "customers")
axes[1].plot(monthly.index, monthly["spend_per_active"], color=C[0], lw=2)
clean(axes[1], "card spend per active customer", "CHF / month")
for a in axes:
    a.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
    a.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
plt.tight_layout()

young bank, still growing: 689 → 1 646 active per month. spend per customer flat at ~1.4–1.6k. raw monthly totals mostly show customer growth → work with shares or per-customer numbers.

In [ ]:
share = (cust[CAT].sum() / cust["total_amount"].sum() * 100).sort_values()
share.index = share.index.str.replace("cat_", "")
share = share[share > 0.5]

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(share.index, share.values, color=C[0])
for i, v in enumerate(share.values):
    ax.annotate(f"{v:.1f}", (v, i), xytext=(4, 0), textcoords="offset points", va="center", color=INK2, fontsize=9)
clean(ax, "where the money goes", "")
ax.set_xlabel("% of total card spend")
ax.grid(axis="y", visible=False)
plt.tight_layout()

shopping, groceries, restaurants, cash, holidays = ~72 %. keep `cash` in mind, comes back in part 2.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].hist(np.log10(cust["total_amount"].clip(lower=1)), bins=40, color=C[0])
clean(axes[0], "total spend per customer", "customers")
axes[0].set_xlabel("log10 CHF")
axes[1].hist(np.log10(cust["n_transactions"]), bins=40, color=C[0])
clean(axes[1], "transactions per customer", "")
axes[1].set_xlabel("log10 transactions")
axes[2].hist(cust["days_active"], bins=40, color=C[0])
clean(axes[2], "days active (last − first tx)", "")
axes[2].set_xlabel("days")
plt.tight_layout()

q = cust["total_amount"].quantile([0.5, 0.9, 0.99]).round(0)
print("total_amount median / p90 / p99:", q.tolist())
print("one-day customers:", f"{(cust['days_active'] == 1).mean():.1%}", "| single transaction:", f"{(cust['n_transactions'] == 1).mean():.1%}")

heavy tail. median 49 tx / 1.9k CHF, top 1 % above 100k. 9 % were active exactly one day. log everything before clustering.

In [ ]:
def counterpart_share(category, groups, by="quarter"):
    d = cp[cp["category"] == category].copy()
    d["grp"] = d["top_counterpart"].map(groups).fillna("other")
    p = d.pivot_table(index=by, columns="grp", values="total_amount", aggfunc="sum").fillna(0)
    return p.div(p.sum(axis=1), axis=0) * 100


groc = counterpart_share("groceries", {"coop": "coop", "migros": "migros",
                                        "lidl": "discounter", "aldi": "discounter", "denner": "discounter"})
fig, ax = plt.subplots(figsize=(8, 3.6))
x = groc.index.to_timestamp()
for i, col in enumerate(["coop", "migros", "discounter"]):
    label_end(ax, x, groc[col], col, C[i])
clean(ax, "grocery wallet, % of grocery card spend per quarter", "%")
ax.set_ylim(0, 35)
plt.tight_layout()

the classic SoW view. coop 25–30, migros 20–25, lidl+aldi+denner 11–14 and creeping up. "other" (volg, spar, kiosks, …) not shown, ~35 %.

In [ ]:
trans = counterpart_share("transport", {
    **{k: "fuel" for k in ["avia", "migrol", "shell", "agrola", "socar", "eni", "tamoil", "coop", "landi"]},
    **{k: "public transport" for k in ["sbb", "fairtiq", "zvv", "vbz"]},
    **{k: "micromobility" for k in ["lime", "tier", "voi"]},
    "uber": "ride hailing", "bolt": "ride hailing",
})
fig, ax = plt.subplots(figsize=(8, 3.6))
x = trans.index.to_timestamp()
for i, col in enumerate(["fuel", "public transport", "ride hailing", "micromobility"]):
    label_end(ax, x, trans[col], col, C[i])
clean(ax, "transport wallet, % of transport card spend per quarter", "%")
plt.tight_layout()

fuel (incl. coop pronto) 40 → 27 %. sbb / fairtiq / zvv flat around 15. micromobility tiny but seasonal (9 users jan 21, 88 oct 22). fuel prices + weather would fit here, but only 36 points.

In [ ]:
lab = cust.merge(labels, on="customer_id")
lab["churned"] = lab["churned"].astype(int)
lab["tenure"] = pd.cut(lab["days_active"], [0, 7, 30, 90, 180, 365, 540, 730, 900, 1000],
                       labels=["≤7d", "8–30d", "1–3m", "3–6m", "6–12m", "12–18m", "18–24m", "24–30m", ">30m"])
ct = lab.groupby("tenure", observed=True)["churned"].agg(["mean", "count"])

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(ct.index.astype(str), ct["mean"] * 100, color=C[0], width=0.7)
for i, (m, n) in enumerate(zip(ct["mean"], ct["count"])):
    ax.annotate(f"n={n}", (i, m * 100), xytext=(0, 3), textcoords="offset points", ha="center", color=INK2, fontsize=8)
clean(ax, "churn rate by tenure", "% churned")
ax.set_ylim(0, 105)
plt.tight_layout()

rule = (lab["days_active"] < 365).astype(int)
print("churn rate:", round(lab["churned"].mean(), 3))
print("rule days_active < 365 → churned, accuracy:", round((rule == lab["churned"]).mean(), 3))

55 % churned. tenure explains most of it. one if-statement gets 74 % accuracy, ML gets ~77 % / AUC 0.85 (tested logreg, RF, boosting). every group will land there. the hand-in won't be the differentiator.

**data quirks**
- `n_transasctions` typo in both sow files
- categories in two spellings (Groceries/groceries, Restaurant/restaurants, Health, Shopping, …), also as duplicate columns in customer_data
- counterpart variants: brezelkonig/brezelkönig, mcdonalds/mcdonald's, amazon/amzn
- same company in several categories (migros in groceries/restaurants/shopping, coop in groceries/transport/health) → category = merchant category of the tx
- 2020-12: one customer, dropped
- currencies cover 96 % of the amount, countries 87 % of tx. rest unmapped
- days_active max 964 but window is 1 095 days → customer extract probably ends ~aug 23. ask lukas
- churn definition unknown. ask lukas

## 2. the case: wallet leakage

**stakeholder: YAPEAL** (retail product / CRM)

share of wallet turned around. not "how much of the grocery wallet goes to coop" but "how much of the customer's *financial* wallet stays at yapeal". the card data shows money leaving to revolut, crypto exchanges and house banks. and those customers leave.

why this one: it's the only topic that uses all three files. `cash` + `savings` are the one category whose counterparts are homogeneous (all financial providers), so it bridges the monthly counterpart data and the customer data.

In [ ]:
fin = cp[cp["category"].isin(["cash", "savings"])]
top = fin.groupby("top_counterpart")["total_amount"].sum().sort_values()
top = top / fin["total_amount"].sum() * 100

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(top.index, top.values, color=[C[3] if n == "other" else C[0] for n in top.index])
for i, v in enumerate(top.values):
    ax.annotate(f"{v:.0f}", (v, i), xytext=(4, 0), textcoords="offset points", va="center", color=INK2, fontsize=9)
clean(ax, "'cash' + 'savings': who gets the money", "")
ax.set_xlabel("% of cash+savings spend")
ax.grid(axis="y", visible=False)
plt.tight_layout()

"cash" is not cash. revolut, binance, coinbase, postfinance, ubs, kantonalbank, twint. that's money moved to other financial providers, 12–17 % of all card spend.

(revolut jumps from `savings` to `cash` in 2022Q1, recategorisation in the source → always read both together)

In [ ]:
dest = {"revolut": "revolut", "binance": "crypto exchanges", "coinbase": "crypto exchanges",
        "postfinance": "traditional banks", "ubs": "traditional banks", "kantonalbank": "traditional banks",
        "twint": "twint"}
leak = fin.assign(grp=fin["top_counterpart"].map(dest)).dropna(subset=["grp"])
leak_q = leak.pivot_table(index="quarter", columns="grp", values="total_amount", aggfunc="sum").fillna(0)
leak_pct = leak_q.div(sow.groupby("quarter")["total_amount"].sum(), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 3.8))
x = leak_pct.index.to_timestamp()
for i, col in enumerate(["revolut", "traditional banks", "crypto exchanges", "twint"]):
    label_end(ax, x, leak_pct[col], col, C[i])
clean(ax, "wallet leakage: card spend going to other financial providers", "% of all card spend")
ax.set_ylim(0, 5)
plt.tight_layout()
leak_pct.round(2)

revolut 3.0 → 4.2 % of *all* card spend. house banks 0.9 → 2.6 % (SNB went positive sep 22, banks pay interest again, yapeal doesn't). crypto peaks at the 2021 btc highs, fades in 2023.

external data that fits here: btc monthly price, SNB policy rate, revolut CH customer numbers (~1 mio by 2025), EUR/CHF.

In [ ]:
rev = fin[fin["top_counterpart"] == "revolut"].groupby("date").agg(cust=("n_customers", "sum"), amt=("total_amount", "sum"))
rev["pct_active"] = rev["cust"] / monthly["active"] * 100
crypto = fin[fin["top_counterpart"].isin(["binance", "coinbase"])].groupby("date")["total_amount"].sum()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].bar(rev.index, rev["pct_active"], width=25, color=C[0])
clean(axes[0], "customers topping up revolut, % of monthly active", "%")
axes[1].bar(crypto.index, crypto / 1000, width=25, color=C[2])
clean(axes[1], "card spend to binance + coinbase", "kCHF / month")
for a in axes:
    a.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
    a.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
plt.tight_layout()
print("revolut users, % of active: mean", round(rev["pct_active"].mean(), 1), "min", round(rev["pct_active"].min(), 1), "max", round(rev["pct_active"].max(), 1))

**1 in 20 customers sends money to revolut every month.** steady, not a one-off. crypto: apr/may 21, nov 21, may/jun 22 = the btc peaks and the luna dip. the wallet follows the market.

In [ ]:
ten = lab[lab["days_active"] > 365].copy()
ten["leak_share"] = ten["cat_savings"] / ten["total_amount"]
ten["bin"] = pd.cut(ten["leak_share"], [-0.01, 0, 0.05, 0.2, 0.5, 1.0], labels=["0 %", "0–5 %", "5–20 %", "20–50 %", ">50 %"])
ladder = ten.groupby("bin", observed=True)["churned"].agg(["mean", "count"])

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(ladder.index.astype(str), ladder["mean"] * 100, color=C[1], width=0.7)
for i, (m, n) in enumerate(zip(ladder["mean"], ladder["count"])):
    ax.annotate(f"n={n}", (i, m * 100), xytext=(0, 3), textcoords="offset points", ha="center", color=INK2, fontsize=8)
clean(ax, "customers active > 1 year: churn vs share of spend sent to savings counterparts", "% churned")
ax.set_xlabel("share of card spend to 'savings' (revolut / coinbase)")
ax.set_ylim(0, 90)
plt.tight_layout()

leakage predicts churn. same tenure (> 1 year), churn goes 29 → 44 → 54 → 70 %. groups get small at the top (n = 20), but the ladder is clean. these customers use yapeal as a feeder account, then leave.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score

X = lab.drop(columns=["customer_id", "churned", "tenure"]).copy()
X["tx_per_day"] = X["n_transactions"] / X["days_active"]
X["amt_per_tx"] = X["total_amount"] / X["n_transactions"]
X["cp_per_tx"] = X["n_counterparts"] / X["n_transactions"]
X["foreign_tx_share"] = lab[[c for c in CTY if c != "country_ch"]].sum(axis=1) / lab["n_transactions"]
for c in CAT:
    X[c + "_share"] = lab[c] / lab["total_amount"]
y = lab["churned"].values

leak_cols = [c for c in X if c.startswith(("cat_cash", "cat_savings"))]
cv = StratifiedKFold(5, shuffle=True, random_state=0)
rf = RandomForestClassifier(400, min_samples_leaf=3, random_state=0, n_jobs=-1)
for name, cols in [("all features", list(X.columns)), ("without leakage", [c for c in X if c not in leak_cols])]:
    p = cross_val_predict(rf, X[cols], y, cv=cv, method="predict_proba")[:, 1]
    print(f"{name:16s} AUC {roc_auc_score(y, p):.3f}  acc {accuracy_score(y, p > 0.5):.3f}")

imp = pd.Series(rf.fit(X, y).feature_importances_, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(imp.index[:12][::-1], imp.values[:12][::-1], color=C[0])
clean(ax, "random forest, top 12 features", "")
ax.grid(axis="y", visible=False)
plt.tight_layout()
ranks = imp.rank(ascending=False).astype(int)
print("rank of leakage features out of", len(imp), ":", ranks[leak_cols].to_dict())

honest take: ~0.85 AUC with or without the leakage features. tenure and activity dominate, leakage sits mid-table. so the case is the *segment* and the money behind it, not model accuracy. the hand-in is this model with a threshold tuned on train.

### research questions

| # | question | type | data | how |
|---|---|---|---|---|
| RQ1 | how much leaves to whom, and how did it develop 21–23? | descriptive | counterpart × month | shares, split into customer growth vs per-customer leakage |
| RQ2 | which behavioural segments exist, which ones leak? | segmentation | customer | RFM-ish features + category mix + intl usage, log-scaled, k-means / HDBSCAN, leakage per cluster |
| RQ3 | is leakage an early churn signal beyond tenure? | predictive | customer + labels | churn model ± leakage features, SHAP. = hand-in |
| RQ4 | what's at stake, what should yapeal do? | prescriptive | all | CLV proxy per cluster (spend/month × expected months from churn model), retention triggers, feature gaps (interest, crypto, FX) |

### external data

- btc monthly price (coingecko / yahoo) ↔ binance + coinbase outflows
- SNB policy rate + savings rates ↔ bank outflows after sep 22
- revolut CH customer numbers (IFZ blog, press) ↔ revolut outflows
- EUR/CHF ↔ holidays / eur spend, revolut as travel card
- covid measures 2021H1 ↔ restaurants / holidays dip (so nobody reads it as a trend)

### story for the presentation

1. "every month one in twenty yapeal customers sends money to revolut"
2. leakage chart → who and how much
3. segments → the feeder segment
4. churn ladder 29 → 70 %
5. what a feeder customer is worth + 3 retention actions

context slide: yapeal has since moved to corporate banking, retail app runs without marketing (IFZ 2026). pitch = "what the 21–23 data already said about why retail customers leave".

### fallbacks

- **grocery wallet under inflation** – coop vs migros vs discounters, deflate with BFS LIK food index, forecast 2024. no customer dimension though
- **transport wallet** – fuel 40 → 27 %, fuel prices (avenergy) + weather (open-meteo). same limit, 36 points
- **gambling** – 17–30 % of entertainment spend to online casinos / betting, ~80 customers a month. one slide under "responsible banking", not a project

### to ask lukas

- how is "churned" defined
- end date of the customer extract
- are the `cash` counterparts outgoing transfers or card top-ups